In [16]:
# Cell 1: 导入库
import pandas as pd
import numpy as np
from pathlib import Path
import librosa
import librosa.display
import matplotlib.pyplot as plt
import soundfile as sf
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings("ignore", category=UserWarning)  # 忽略 librosa 的一些警告

In [ ]:
# Cell 2: 配置（你可以在这里随时调整）
CONFIG = {
    "sr": 48000,
    "clip_duration_sec": 60.0,
    "instance_duration_sec": 1.0,
    "n_mfcc": 40,
    "hop_length": 512,
    "n_fft": 2048,
    "fmax": 8000,
    "include_delta": True,
    "include_delta_delta": True,
    "cmvn": True,               # per-clip 归一化
    "output_root": Path(r"D:\Project_Github\audio_click_mil\data\processed\features_1s"),
    "npy_dir": Path("mfcc_npy"),
    "img_dir": Path("mfcc_images"),
    "label_csv": Path(r"D:\Project_Github\audio_click_mil\data\processed\instance_labels\instance_labels.csv"),
    "dpi": 150,
}

# 创建输出目录
CONFIG["output_root"].mkdir(parents=True, exist_ok=True)
(CONFIG["output_root"] / CONFIG["npy_dir"]).mkdir(exist_ok=True)
(CONFIG["output_root"] / CONFIG["img_dir"]).mkdir(exist_ok=True)
CONFIG["label_csv"].parent.mkdir(parents=True, exist_ok=True)

In [18]:
# Cell 3: 读取两个 csv
clip_df = pd.read_csv(r"D:\Project_Github\audio_click_mil\data\processed\metadata\clip_labels.csv")
anno_df = pd.read_csv(r"D:\Project_Github\audio_click_mil\data\raw\annotations\click_annotations.csv")
anno_df['Ori_file_num(No.)'] = anno_df['Ori_file_num(No.)'].astype(int)

In [19]:
# Cell 4: 核心函数 - 处理单个 1min clip
def process_one_clip(
    clip_row,
    anno_df,
    config=CONFIG
):
    clip_path = Path(r"D:\Project_Github\audio_click_mil\data\processed\clips_1min") / clip_row['clip_filename']
    if not clip_path.exists():
        print(f"文件不存在，跳过: {clip_path}")
        return []

    # 读取音频 (应该已经是 48kHz float)
    y, sr = librosa.load(clip_path, sr=None, mono=True)
    if sr != config["sr"]:
        print(f"采样率不匹配: {clip_path} -> 重采样")
        y = librosa.resample(y, orig_sr=sr, target_sr=config["sr"])
        sr = config["sr"]

    duration = len(y) / sr
    n_instances = int(duration // config["instance_duration_sec"])

    # 获取这个 clip 对应的原始音频的所有 click 区间
    orig_audio = clip_row['original_audio']
    file_num = int(orig_audio.replace("Ori_Recording_", "").replace(".wav", ""))
    clicks_this_file = anno_df[anno_df['Ori_file_num(No.)'] == file_num]
    click_intervals = list(zip(clicks_this_file['Train_start(s)'], clicks_this_file['Train_end(s)']))

    # 这个 clip 在原音频中的起始时间
    clip_start_sec = clip_row['start_sec']

    records = []

    for i in range(n_instances):
        inst_start = i * config["instance_duration_sec"]
        inst_end   = (i + 1) * config["instance_duration_sec"]

        # 转换为原音频绝对时间
        abs_start = clip_start_sec + inst_start
        abs_end   = clip_start_sec + inst_end

        # 判断是否有 click 区间与 [abs_start, abs_end) 重叠
        has_click = any(
            not (c_end <= abs_start or c_start >= abs_end)
            for c_start, c_end in click_intervals
        )
        label = 1 if has_click else 0

        # 切片波形
        start_sample = int(inst_start * sr)
        end_sample   = int(inst_end * sr)
        segment = y[start_sample:end_sample]

        # 计算 MFCC + deltas
        mfcc = librosa.feature.mfcc(
            y=segment, sr=sr,
            n_mfcc=config["n_mfcc"],
            hop_length=config["hop_length"],
            n_fft=config["n_fft"],
            fmax=config["fmax"]
        )

        if config["include_delta"]:
            delta = librosa.feature.delta(mfcc)
            mfcc = np.vstack([mfcc, delta])
        if config["include_delta_delta"]:
            dd = librosa.feature.delta(mfcc, order=2)
            mfcc = np.vstack([mfcc, dd])

        mfcc = mfcc.T  # (n_frames, n_features)

        # per-clip CMVN（注意：这里用整个 1min clip 的统计？还是每个 1s 单独归一化？）
        # 推荐：每个 1s 独立做 CMVN（更独立，避免 clip 内部偏差影响）
        if config["cmvn"]:
            mean = np.mean(mfcc, axis=0, keepdims=True)
            std = np.std(mfcc, axis=0, keepdims=True) + 1e-8
            mfcc = (mfcc - mean) / std

        # 保存 npy
        stem = clip_row['clip_filename'].replace(".wav", "")
        inst_name = f"{stem}_{i:03d}"
        npy_path = config["output_root"] / config["npy_dir"] / f"{inst_name}.npy"
        np.save(npy_path, mfcc)

        # 画图并保存
        fig_title = f"{stem} | inst {i:03d} | label:{label}"
        img_path = config["output_root"] / config["img_dir"] / f"{inst_name}.png"

        plt.figure(figsize=(10, 4))
        librosa.display.specshow(
            mfcc.T,  # (n_features, n_frames)
            x_axis='time',
            hop_length=config["hop_length"],
            sr=sr,
            cmap='viridis',
            fmax=config["fmax"]
        )
        plt.colorbar(format='%+2.0f')
        plt.title(fig_title)
        plt.tight_layout()
        plt.savefig(img_path, dpi=config["dpi"], bbox_inches='tight')
        plt.close()

        # 记录
        records.append({
            "instance_filename": f"{inst_name}.npy",
            "clip_filename": clip_row['clip_filename'],
            "instance_idx": i,
            "start_sec_in_clip": round(inst_start, 3),
            "end_sec_in_clip": round(inst_end, 3),
            "start_sec_abs": round(abs_start, 3),
            "end_sec_abs": round(abs_end, 3),
            "label": label,
            "original_audio": orig_audio
        })

    return records

In [34]:
# ===================== 新增 Cell：按每个 long audio 内部平衡正负包 =====================

# 假设你已经读取了 clip_df（如果没有，先运行下面这行）
clip_df = pd.read_csv(r"D:\Project_Github\audio_click_mil\data\processed\metadata\clip_labels.csv")

import pandas as pd
from pathlib import Path

# ------------------ 配置（你可以调整） ------------------
TARGET_RATIO = 1.5          # 每个音频里，负包最多保留到 正包数量 × 这个倍数
MAX_NEG_PER_FULL_NEG_AUDIO = 10  # 如果一个音频完全没有正包，最多保留多少负包（防止全丢）

# -------------------------------------------------------

print("原始整体分布：")
print(clip_df['has_click'].value_counts(normalize=True))
print(clip_df['has_click'].value_counts())

# 按 original_audio 分组进行局部平衡
grouped = clip_df.groupby("original_audio")

balanced_records = []

for orig_audio, group in grouped:
    pos = group[group["has_click"] == 1]
    neg = group[group["has_click"] == 0]
    
    n_pos = len(pos)
    n_neg = len(neg)
    
    print(f"{orig_audio: <25} | 正包: {n_pos:3d} | 负包: {n_neg:4d} → ", end="")
    
    if n_pos == 0:
        # 全负音频：可选保留少量负样本，避免完全丢弃这个音频
        if n_neg > 0:
            kept_neg = neg.sample(min(MAX_NEG_PER_FULL_NEG_AUDIO, n_neg))
            balanced_records.extend(kept_neg.to_dict('records'))
            print(f"保留 {len(kept_neg)} 个负包 (全负音频)")
        else:
            print("无样本，跳过")
        continue
    
    # 有正包的音频：保留所有正包 + 采样负包
    n_keep_neg = min(n_neg, int(n_pos * TARGET_RATIO))
    
    if n_keep_neg < n_neg:
        kept_neg = neg.sample(n_keep_neg)
        print(f"保留 {n_keep_neg} 个负包 (采样 {n_neg - n_keep_neg} 个)")
    else:
        kept_neg = neg
        print(f"负包已少于 {int(n_pos * TARGET_RATIO)}，全保留")
    
    balanced_records.extend(pos.to_dict('records'))
    balanced_records.extend(kept_neg.to_dict('records'))

# 生成平衡后的 DataFrame
balanced_df = pd.DataFrame(balanced_records)

print("\n" + "="*60)
print("平衡后整体分布：")
print(balanced_df['has_click'].value_counts(normalize=True))
print(balanced_df['has_click'].value_counts())

# 可选：保存平衡后的 clip 列表（供后续 splits 使用）
balanced_clip_list_path = Path(r"D:\Project_Github\audio_click_mil\data\processed\balanced_clip_list.csv")
balanced_df.to_csv(balanced_clip_list_path, index=False)
print(f"\n已保存平衡后的 clip 列表到：{balanced_clip_list_path}")

# 如果你后续要用这个 balanced_df 来生成 train/val/test.txt
# 就可以基于 balanced_df 继续划分，而不是用原始 clip_df

原始整体分布：
has_click
0    0.689655
1    0.310345
Name: proportion, dtype: float64
has_click
0    100
1     45
Name: count, dtype: int64
Ori_Recording_01.wav      | 正包:   5 | 负包:   24 → 保留 7 个负包 (采样 17 个)
Ori_Recording_02.wav      | 正包:  21 | 负包:    8 → 负包已少于 31，全保留
Ori_Recording_03.wav      | 正包:   5 | 负包:   24 → 保留 7 个负包 (采样 17 个)
Ori_Recording_04.wav      | 正包:   5 | 负包:   24 → 保留 7 个负包 (采样 17 个)
Ori_Recording_05.wav      | 正包:   9 | 负包:   20 → 保留 13 个负包 (采样 7 个)

平衡后整体分布：
has_click
1    0.517241
0    0.482759
Name: proportion, dtype: float64
has_click
1    45
0    42
Name: count, dtype: int64

已保存平衡后的 clip 列表到：D:\Project_Github\audio_click_mil\data\processed\balanced_clip_list.csv


In [36]:
# Cell: 数据集划分（train / val / test）

import pandas as pd
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit, train_test_split

# ────────────────────────────────────────────────
# 配置（可调整）
# ────────────────────────────────────────────────
SPLIT_ROOT = Path(r"D:\Project_Github\audio_click_mil\data\processed\splits")
SPLIT_ROOT.mkdir(parents=True, exist_ok=True)

train_ratio = 0.8
val_ratio   = 0.1
test_ratio  = 0.1

random_state = 42

output_files = {
    'train': SPLIT_ROOT / "train.txt",
    'val':   SPLIT_ROOT / "val.txt",
    'test':  SPLIT_ROOT / "test.txt"
}

# ────────────────────────────────────────────────
# 读取平衡后的 clip 列表
# ────────────────────────────────────────────────
balanced_df = pd.read_csv(r"D:\Project_Github\audio_click_mil\data\processed\balanced_clip_list.csv")

print("平衡后 clip 数量:", len(balanced_df))
print("正样本比例:", balanced_df['has_click'].mean())

# ────────────────────────────────────────────────
# 按原始长音频分组划分（GroupShuffleSplit）
# ────────────────────────────────────────────────

# ────────────────────────────────────────────────
# 按原始音频做三方划分（更稳健，避免剩余样本太少）
# ────────────────────────────────────────────────

unique_audios = balanced_df['original_audio'].unique()
print(f"共有 {len(unique_audios)} 个独立的长音频文件")

# 先把原始音频分成三部分
# 因为音频数量少（只有5个），我们手动或用 train_test_split 做分层
# 这里用分层 + group 的方式

from sklearn.model_selection import train_test_split

# 先分出 test (10%)
audio_train_val, audio_test = train_test_split(
    unique_audios,
    test_size=test_ratio,
    random_state=random_state,
    stratify=None  # 音频数量少，暂不分层
)

# 再从剩余中分出 train 和 val (80% vs 10% → 8:1)
audio_train, audio_val = train_test_split(
    audio_train_val,
    test_size=val_ratio / (train_ratio + val_ratio),
    random_state=random_state,
    stratify=None
)

print(f"Train 音频数: {len(audio_train)}")
print(f"Val   音频数: {len(audio_val)}")
print(f"Test  音频数: {len(audio_test)}")

# 根据音频名过滤 df
train_df = balanced_df[balanced_df['original_audio'].isin(audio_train)]
val_df   = balanced_df[balanced_df['original_audio'].isin(audio_val)]
test_df  = balanced_df[balanced_df['original_audio'].isin(audio_test)]

# 检查比例
print("\n划分后正样本比例：")
print(f"Train: {train_df['has_click'].mean():.4f} ({len(train_df)} clips)")
print(f"Val:   {val_df['has_click'].mean():.4f} ({len(val_df)} clips)")
print(f"Test:  {test_df['has_click'].mean():.4f} ({len(test_df)} clips)")

# ────────────────────────────────────────────────
# 保存文件名列表（不带 .wav 后缀）
# ────────────────────────────────────────────────
def save_split_txt(df, filepath):
    stems = df['clip_filename'].str.replace('.wav', '', regex=False).tolist()
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write('\n'.join(stems))
    print(f"已保存 {len(stems)} 条到 {filepath}")

save_split_txt(train_df, output_files['train'])
save_split_txt(val_df,   output_files['val'])
save_split_txt(test_df,  output_files['test'])

print("\n划分完成！")
print("文件保存位置：", SPLIT_ROOT)

平衡后 clip 数量: 87
正样本比例: 0.5172413793103449
共有 5 个独立的长音频文件
Train 音频数: 3
Val   音频数: 1
Test  音频数: 1

划分后正样本比例：
Train: 0.4130 (46 clips)
Val:   0.4167 (12 clips)
Test:  0.7241 (29 clips)
已保存 46 条到 D:\Project_Github\audio_click_mil\data\processed\splits\train.txt
已保存 12 条到 D:\Project_Github\audio_click_mil\data\processed\splits\val.txt
已保存 29 条到 D:\Project_Github\audio_click_mil\data\processed\splits\test.txt

划分完成！
文件保存位置： D:\Project_Github\audio_click_mil\data\processed\splits


In [37]:
# Cell 8: 全量生成 instance-level（优化版：只处理 train/val/test 中的 clip）

# 先读取三个 split 文件，得到所有需要的 stem
needed_stems = set()

for split_name in ['train', 'val', 'test']:
    split_path = r"D:\Project_Github\audio_click_mil\data\processed\splits" + f"/{split_name}.txt"
    if Path(split_path).exists():
        with open(split_path, 'r', encoding='utf-8') as f:
            stems = {line.strip() for line in f if line.strip()}
            needed_stems.update(stems)
    else:
        print(f"警告：{split_path} 不存在，跳过")

print(f"需要处理的 clip 数量：{len(needed_stems)} / {len(clip_df)}")

# 过滤 clip_df，只保留需要的 clip
process_df = clip_df[clip_df['clip_filename'].str.replace('.wav', '', regex=False).isin(needed_stems)]

all_records = []

for _, row in tqdm(process_df.iterrows(), total=len(process_df), desc="处理实例级特征"):
    records = process_one_clip(row, anno_df, CONFIG)
    all_records.extend(records)

# 保存 instance_labels.csv
instance_df = pd.DataFrame(all_records)
instance_df.to_csv(CONFIG["label_csv"], index=False)
print(f"全部完成，已保存 {len(instance_df)} 条实例标签到 {CONFIG['label_csv']}")

# 可选：统计正负实例比例
if not instance_df.empty:
    print("\n实例级正负比例：")
    print(instance_df['label'].value_counts(normalize=True))

<string>:7: SyntaxWarning: invalid escape sequence '\{'
<>:7: SyntaxWarning: invalid escape sequence '\{'
<string>:7: SyntaxWarning: invalid escape sequence '\{'
<>:7: SyntaxWarning: invalid escape sequence '\{'
C:\Users\Chico\AppData\Local\Temp\ipykernel_26328\3032866095.py:7: SyntaxWarning: invalid escape sequence '\{'
  split_path = r"D:\Project_Github\audio_click_mil\data\processed\splits" + f"\{split_name}.txt"


需要处理的 clip 数量：87 / 145


处理实例级特征:   6%|▌         | 5/87 [01:24<23:11, 16.97s/it]
C:\Users\Chico\AppData\Local\Temp\ipykernel_26328\3032866095.py:7: SyntaxWarning: invalid escape sequence '\{'
  split_path = r"D:\Project_Github\audio_click_mil\data\processed\splits" + f"\{split_name}.txt"


KeyboardInterrupt: 